In [71]:
import time
import numpy as np
import sklearn
from sklearn.neighbors import KDTree, BallTree, radius_neighbors_graph
from sklearn.metrics.pairwise import euclidean_distances
from scipy.sparse import csr_array
from scipy.io import mmwrite, mmread

import cppimport
import cppimport.import_hook
from extensions.brute_force import *

class BruteForce(object):

    def __init__(self, points):
        self.points = points
        self.n = self.points.shape[0]
        self.d = self.points.shape[1]
    
    def query(self, X, k=1, return_distance=True, num_threads=4):
        assert X.shape[1] == self.d
        m = X.shape[0]
        dists = np.empty((m,k), dtype=np.float64)
        inds = np.empty((m,k), dtype=np.int64)
        brute_force_query(self.points, X, dists, inds, num_threads)
        if return_distance: return dists, inds
        else: return inds
    
    def query_radius_counts(self, X, r, num_threads=4):
        assert X.shape[1] == self.d
        return brute_force_query_radius_count_only(self.points, X, r, num_threads)
    
    def radius_neighbors_graph(self, radius, num_threads=4):
        rowptrs, colids, dists = brute_force_query_radius_neighbors(self.points, self.points, radius, num_threads)
        return csr_array((dists, colids, rowptrs), shape=(self.n,self.n))

        

n, d = 1000, 8
points = np.random.uniform(0,1,size=(n,d))

brute_force = BruteForce(points)
tree = KDTree(points)


In [85]:
graph = radius_neighbors_graph(points, 0.7, mode="distance",include_self=True)

In [86]:
graph

<1000x1000 sparse matrix of type '<class 'numpy.float64'>'
	with 45362 stored elements in Compressed Sparse Row format>

In [87]:
graph2 = brute_force.radius_neighbors_graph(0.7)

In [88]:
graph2

<1000x1000 sparse array of type '<class 'numpy.float64'>'
	with 45362 stored elements in Compressed Sparse Row format>

In [91]:
print(np.allclose(graph.todense(), graph2.todense()))

True
